In [3]:
import cupt_parser
import lambda_css_lexicon
import lambda_css
import mwe_lexicon
import oa_indices

parseme_path = 'data/parseme'

In [ ]:
df_test2 = cupt_parser.setup_data_noTT(
	parseme_path+'/1.2/FR/test.blind.cupt.wesh'
)

x= cupt_parser.get_mwes(df_test2)

In [ ]:
TT_train, df_train = cupt_parser.setup_data(
		parseme_path+'/1.2/FR/train.cupt'
	)

TT_test, df_test = cupt_parser.setup_data(
		parseme_path+'/1.2/FR/test.cupt'
	)

In [6]:
lcss_cl = lambda_css.LambdaCSS.get_lCSS({'lemma' : True, 'deprel': False})
lcss_lex_formalism = mwe_lexicon.Lexicon_formalism.concretize(
	lcss_cl,
	lambda_css_lexicon.extract_lcss_lexicon,
	lambda_css_lexicon.lexicon_matches_in_sentences
)
lcss_lex = lcss_lex_formalism.instantiate(TT_train)


C:\Users\lionbouton\Code\phd_code\lambda_css\src\lambda_css\lambdacss.py:284: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  lambda x: lCSS_cl.from_mwe_mdg(x[1], x[0]), axis=1


In [7]:
truth = cupt_parser.get_mwes(df_test)

In [8]:
pred = lcss_lex.match(df_test)

In [9]:
oa_indices.full_eval(truth, pred)


{'p': 0.7881286067600989,
 'r': 0.70345842531273,
 'f': 0.7433903576982892,
 'richness': 373.0,
 'N1': 200.6407851805746,
 'normalize_r': 0.3901673640167364,
 'H': 7.648471088830268,
 'J': 0.8952876741833037,
 'e10': 0.5379109522267416,
 'e21': 0.444311923768969,
 'e20': 0.2390002500002615,
 'F10': 0.536668777367136,
 'F21': 0.4415284841239665,
 'F20': 0.23695455174757404,
 'E_heip': 0.536668777367136,
 'E_1mD': 0.9914406020383443,
 'E_1dD': 0.2390002500002615,
 'E_lnD': 0.758292370784635,
 'G_21': 0.08607483101824848,
 'O': 0.61040192042357,
 'E_x': 0.609354613758042,
 'E_mcl': 0.9429097125253992,
 'E_prime': np.float64(0.5029894444008219),
 'E_var': np.float64(0.6853245754913218),
 '1/S': np.float64(1.4260803796890098)}

In [7]:
re.match(r'(\d+(?:-\d+)?)', '12	Tenue	tenir	VERB	_	Gender=Fem|Number=Sing|Tense=Past|VerbForm=Part	6	acl	_	_	1:LVC.full')

NameError: name 're' is not defined

In [89]:
set(matches.loc[892].sort_index().loc[4].index)

{94, 95}

In [16]:

def f(matches, file_path):
	s_id = -1
	og_file = open(file_path, 'r', encoding='utf-8')
	new_file = open(file_path+'.wesh', 'w', encoding='utf-8')
	id_map = {}
	flag = False
	@lru_cache
	def intern(s_id):
		res = matches.loc[s_id].sort_index()
		return res, set(res.index.droplevel(0))

	for line in og_file:
		#get number of the token or none if not a token line
		
		tok_id = re.match(r'(\d+(?:-\d+)?)', line)

		# copy the line if not a token line (comment line, empty line...)
		if tok_id is None:
			new_file.write(line)
			continue

		tok_id = tok_id[0]

		# incr the id of the current sentence if we begin a new sentence
		if tok_id == '1' or tok_id.startswith('1-'):
			if not flag:
				s_id +=1
				id_map = {}
			if flag:
				flag = False
			if tok_id.startswith('1-'):
				flag = True
			
		try:
			tok_id = int(tok_id)
		except:
			pass

		# copy the line if the current sentence does not contain any match
		if s_id not in matches.index:
			new_file.write(line)
			continue

		sent_matches, mwe_ids = intern(s_id)
		
		# copy the line if the token does not contain any match
		if tok_id not in sent_matches.index:
			new_file.write(line)
			continue
		# print(s_id, tok_id)
		split_line = line.split('\t')
		
		sent_matches.loc[tok_id]
		s = ''
		for mwe_id in set(sent_matches.loc[tok_id].index):
			if mwe_id not in id_map:
				id_map[mwe_id] = len(id_map)+1
				if s:
					s+=';'
				s += f'{id_map[mwe_id]}:_'
			else:
				if s:
					s+=';'
				s += f'{id_map[mwe_id]}'
		split_line[-1] = s

		# print(s_id, tok_id, s)
		new_file.write('\t'.join(split_line)+'\n')
	og_file.close()
	new_file.close()

matches = lcss_lex.match(df_test, lambdacss.sort_column(df_test))
f(matches, 'untracked/1.2/FR/test.blind.cupt')

		
			

In [3]:
seq_lex_formalism = lexicons.Lexicon_formalism.concretize(
	lexicons.Seq_rep.concretize(['lemma'], lexicons.disc_fs[0]),
	lexicons.extract_pattern_from_data,
	lexicons.SeqRep_match
)
seq_lex = seq_lex_formalism.instantiate(df_train)

In [4]:
sid_test = lambdacss.sort_column(df_test)

In [132]:
# x = list(lex.content)[6]
# ress = [[]]*len(x.components)
# for n, component in enumerate(x.components):
# 	for k,v in component.items():
# 		if not ress[n] :
# 			ress[n] = set(lexicons.sorted_index_loc(sid[k], v, sid['id'].index))
# 		else:
# 			ress[n] = ress[n].intersection(
# 				set(lexicons.sorted_index_loc(sid[k], v, sid['id'].index))
# 			)

t = perf_counter()
# matches = [
# 	[
# 		list(b.index)
# 		# [v.name for k, v in b.iterrows()]
# 		for _, b in df.loc[list(a)].groupby(level=0)
# 	] for a in ress
# ]
matches = [[
		list(v)
		for k, v in groupby(sorted(a, key= lambda x: x[0]), key= lambda x: x[0])
	] for a in ress
]
t = perf_counter() - t
print(t)

# sentences= [defaultdict(list) for _ in range(len(self.components))]
# for n, x in enumerate(matches):
# 	for y in x:
# 		sentences[n][y[0][0]]+=y

# res=[]
# keys = set.intersection(*[set(x.keys()) for x in sentences])
# # print(sentences)
# for k in keys:
# 	for prod in product(*[x[k] for x in sentences]):
# 		# print(prod)
# 		if [e[1] for e in prod] != sorted([e[1] for e in prod]):
# 			# print([e[1] for e in prod])
# 			continue

# 		prod = [df.loc[e] for e in prod]
# 		# print(prod)				
# 		for insertion, (a, b) in zip(self.insertions, zip(prod[:-1],prod[1:])):
# 			if b['id'] - a['id'] > 1:
# 				# print(df.loc[a.name[0]].loc[a['id']+1:b['id']-1])
# 				try:
# 					tmp = self.__class__.handle_discontinuities(df.loc[a.name[0]].loc[a['id']+1:b['id']-1])
# 					if insertion != tmp:
# 						break
# 				except InvalidInsertion:
# 					break
# 		else:
# 			res.append(pd.DataFrame(prod))
# # print(res)
# return res	

0.010406000000330096


In [ ]:
truth = cupt_parser.get_mwes(df_test)
pred = seq_lex.match(df_test, lambdacss.sort_column(df_test))

In [7]:
lexicons.evaluate(
	truth,
    pred
)

{'p': 0.716304347826087,
 'r': 0.48491537895511405,
 'f': 0.5783238262395788,
 'richness': 233,
 'normalize_r': 0.3535660091047041,
 'e10': 0.5465953666117094,
 'e21': 0.4922704589001213,
 '1/S': 1.4219326288781857}

In [ ]:
langs = ['FR'] #['DE', 'EL', 'EU', 'FR', 'FR_sequoia', 'GA', 'HE', 'HI', 'IT', 'PL', 'PT', 'RO', 'SV', 'TR', 'TR2', 'ZH']
res = {}
for lang in langs:
	res[lang] = {}
	TT_train, df_train = cupt_parser.setup_data(
		'untracked/1.2/'+lang+'/traindev.cupt'
	)

	TT_test, df_test = cupt_parser.setup_data(
			'untracked/1.2/'+lang+'/test.cupt'
		)
	truth = cupt_parser.get_mwes(df_test)
	sid_test = cupt_parser.sort_column(df_test)
	for lex_for in lexicons.disc_fs:
		seq_lex_formalism = lexicons.Lexicon_formalism.concretize(
			lexicons.Seq_rep.concretize(['lemma'], lex_for),
			lexicons.extract_pattern_from_data,
			lexicons.SeqRep_match
		)
		seq_lex = seq_lex_formalism.instantiate(df_train)
		name = seq_lex.__class__.T_cl.handle_discontinuities.__doc__
		res[lang][name] = {}
		pred = seq_lex.match(df_test, sid_test)
		for k, v in lexicons.evaluate(truth,pred).items():
			res[lang][name][k] = v


In [37]:
res

{'FR': {'contiguous': {'p': 0.716304347826087,
   'r': 0.48491537895511405,
   'f': 0.5783238262395788,
   'richness': 233,
   'normalize_r': 0.3535660091047041,
   'e10': 0.5465953666117094,
   'e21': 0.4922704589001213,
   '1/S': 1.4219326288781857},
  '1 pos': {'p': 0.720969089390142,
   'r': 0.6350257542310522,
   'f': 0.6752738654147105,
   'richness': 329,
   'normalize_r': 0.38122827346465815,
   'e10': 0.5282242808424161,
   'e21': 0.4412187340789247,
   '1/S': 1.3960367836885554},
  'list lemma': {'p': 0.7207903780068728,
   'r': 0.6173657100809419,
   'f': 0.6650812524772096,
   'richness': 323,
   'normalize_r': 0.38498212157330153,
   'e10': 0.5274519188339621,
   'e21': 0.4387589134371614,
   '1/S': 1.3873492747892042}}}

In [41]:
[mwe.components for mwe in seq_lex.content if len(mwe.components) != len(set(comp['lemma'] for comp in mwe.components))]

[[{'lemma': 'couper'},
  {'lemma': 'le'},
  {'lemma': 'herbe'},
  {'lemma': 'sous'},
  {'lemma': 'le'},
  {'lemma': 'pied'}],
 [{'lemma': 'remettre'},
  {'lemma': 'le'},
  {'lemma': 'main'},
  {'lemma': 'à'},
  {'lemma': 'le'},
  {'lemma': 'pâte'}],
 [{'lemma': 'ce'},
  {'lemma': 'être'},
  {'lemma': 'le'},
  {'lemma': 'monde'},
  {'lemma': 'à'},
  {'lemma': 'le'},
  {'lemma': 'envers'}],
 [{'lemma': 'le'},
  {'lemma': 'hasard'},
  {'lemma': 'faire'},
  {'lemma': 'bien'},
  {'lemma': 'le'},
  {'lemma': 'chose'}],
 [{'lemma': 'joindre'},
  {'lemma': 'le'},
  {'lemma': 'geste'},
  {'lemma': 'à'},
  {'lemma': 'le'},
  {'lemma': 'parole'}],
 [{'lemma': 'tomber'}, {'lemma': 'nez'}, {'lemma': 'à'}, {'lemma': 'nez'}],
 [{'lemma': 'faire'},
  {'lemma': 'le'},
  {'lemma': 'tour'},
  {'lemma': 'de'},
  {'lemma': 'le'},
  {'lemma': 'monde'}],
 [{'lemma': 'séparer'},
  {'lemma': 'le'},
  {'lemma': 'grain'},
  {'lemma': 'de'},
  {'lemma': 'le'},
  {'lemma': 'ivraie'}],
 [{'lemma': 'mettre'},
  {'le

In [70]:
df = df_test


In [71]:
mwe = df[df["parseme:mwe"].str.match(r"[1-9]") == True]
index_mwe = df["parseme:mwe"].str.match(r"[1-9]") == True
mwe = df[index_mwe]
mwe_rest = mwe.copy()
mwe_rest["parseme:mwe"] = mwe_rest["parseme:mwe"].str.split(":").apply(lambda x: x[0])
per_sentence = mwe_rest.groupby(["sentence_id", "parseme:mwe"])
out = per_sentence["lemma"].transform(lambda x: x.str.cat(sep = "_"))
mwe_rest["lemma"] = out
df.loc[index_mwe, "lemma"] = mwe_rest["lemma"]
out = df.drop(df[df["parseme:mwe"].str.match(r"[1-9]$")].index)


KeyError: "['sentence_id'] not in index"

In [72]:
mwe_rest.loc[754]
# out_lemmas

In [62]:
mwe_rest.loc[754]

,id,form,lemma,upos,xpos,head,deprel,parseme:mwe,Definite,Gender,...,Person,Mood,Polarity,NumType,Poss,Voice,Typo,Reflex,Foreign,Number[psor]
token_id,,,,,,,,,,,,,,,,,,,,,
20,20,fait,faire,VERB,None,7.0,conj,1,NaN,Masc,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,21,l',le,DET,None,22.0,det,1;2,Def,Masc,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,22,objet,objet,NOUN,None,20.0,obj,1;2,NaN,Masc,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24,24,adaptations,adaptation,NOUN,None,22.0,nmod,2,NaN,Fem,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
